In [1]:
import torch
import transformers
import sklearn
import wandb

print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Sklearn version:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch version: 2.9.1
Transformers version: 4.57.3
Sklearn version: 1.8.0
CUDA available: False


In [2]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import wandb
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [3]:
dataset = load_dataset("imdb")
dataset


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [4]:
small_train = dataset["train"].shuffle(seed=42).select(range(4000))  
small_test = dataset["test"].shuffle(seed=42).select(range(1000))

len(small_train), len(small_test)


(4000, 1000)

In [5]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

encoded_train = small_train.map(tokenize_batch, batched=True)
encoded_test = small_test.map(tokenize_batch, batched=True)

encoded_train = encoded_train.remove_columns(["text"])
encoded_test = encoded_test.remove_columns(["text"])

encoded_train.set_format(type="torch")
encoded_test.set_format(type="torch")

encoded_train[0]


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'label': tensor(1),
 'input_ids': tensor([  101,  2045,  2003,  2053,  7189,  2012,  2035,  2090,  3481,  3771,
          1998,  6337,  2099,  2021,  1996,  2755,  2008,  2119,  2024,  2610,
          2186,  2055,  6355,  6997,  1012,  6337,  2099,  3504, 15594,  2100,
          1010,  3481,  3771,  3504,  4438,  1012,  6337,  2099, 14811,  2024,
          3243,  3722,  1012,  3481,  3771,  1005,  1055,  5436,  2024,  2521,
          2062,  8552,  1012,  1012,  1012,  3481,  3771,  3504,  2062,  2066,
          3539,  8343,  1010,  2065,  2057,  2031,  2000,  3962, 12319,  1012,
          1012,  1012,  1996,  2364,  2839,  2003,  5410,  1998,  6881,  2080,
          1010,  2021,  2031,  1000, 17936,  6767,  7054,  3401,  1000,  1012,
          2111,  2066,  2000, 12826,  1010,  2000,  3648,  1010,  2000, 16157,
          1012,  2129,  2055,  2074,  9107,  1029,  6057,  2518,  2205,  1010,
          2111,  3015,  3481,  3771,  3504,  2137,  2021,  1010,  2006,  1996,
          2060,  2

In [6]:
train_loader = DataLoader(encoded_train, batch_size=16, shuffle=True)
test_loader = DataLoader(encoded_test, batch_size=32)

len(train_loader), len(test_loader)


(250, 32)

In [7]:
wandb.login()  


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/snithshibu/.netrc
wandb: Currently logged in as: snithshibu (snithshibu-mar-baselios-college-of-engineering-and-techn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
num_labels = 2  

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
num_epochs = 2

total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

wandb.init(
    project="bert-sentiment-imdb",
    config={
        "model_name": model_name,
        "batch_size": 16,
        "lr": 2e-5,
        "epochs": num_epochs,
        "max_length": 128
    }
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked

In [9]:
EPOCHS = num_epochs

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - train"):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["label"]
        )
        loss = outputs.loss
        logits = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()
        labels = batch["label"].detach().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

        wandb.log({"train_batch_loss": loss.item()})

    avg_train_loss = train_loss / len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    train_f1 = f1_score(all_labels, all_preds, average="weighted")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "train_f1": train_f1,
    })

    print(f"Epoch {epoch+1}: loss={avg_train_loss:.4f}, acc={train_acc:.4f}, f1={train_f1:.4f}")


Epoch 1/2 - train:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 1: loss=0.4705, acc=0.7482, f1=0.7482


Epoch 2/2 - train:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 2: loss=0.2243, acc=0.9117, f1=0.9117


In [10]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test"):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1).detach().cpu().numpy()
        labels = batch["label"].detach().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average="weighted")
cm = confusion_matrix(all_labels, all_preds)

print("Test accuracy:", test_acc)
print("Test F1 (weighted):", test_f1)
print("Confusion matrix:\n", cm)

wandb.log({
    "test_accuracy": test_acc,
    "test_f1": test_f1,
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=all_labels,
        preds=all_preds,
        class_names=["negative", "positive"],
    )
})


Test:   0%|          | 0/32 [00:00<?, ?it/s]

Test accuracy: 0.858
Test F1 (weighted): 0.857892613206679
Confusion matrix:
 [[420  92]
 [ 50 438]]


In [11]:
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))
        return id2label[pred_id], probs

print(predict_sentiment("This movie was absolutely fantastic!"))
print(predict_sentiment("Worst film I have ever seen."))


('positive', array([0.02939503, 0.970605  ], dtype=float32))
('negative', array([0.9667459 , 0.03325405], dtype=float32))


In [13]:
save_dir = "bert_imdb_finetuned"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

save_dir


'bert_imdb_finetuned'